# 03. Model Validation: Baseline Claim Before and After

Compares the claim made before any model was run, the rule based archetype distribution from the baseline notebook, against the claim made after running KMeans, the archetype_kmeans distribution from the clustering notebook, and checks whether the after claim actually holds up against the before claim it is supposed to explain.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.feature_selection import f_classif

# the five behavioral features the final KMeans model clusters on, shared by every
# notebook downstream of the model fit so the feature set cannot silently drift between
# the clustering, validation, and leakage audit notebooks
FEATURE_COLS = ["ctr", "avg_position", "days_since_last_update", "total_impressions_log", "engagement_rate_filled"]

page_level = pd.read_csv("../interim/page_level.csv")
train = pd.read_csv("../interim/train.csv")
test = pd.read_csv("../interim/test.csv")

RAW_COLS = ["total_impressions", "total_clicks", "ctr", "avg_position", "days_since_last_update"]
SIL_SAMPLE_SIZE = 5000

print(f"pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}")
print("every random draw below is seeded at 42, rerunning this notebook against the same interim CSVs reproduces the same numbers")

## Claim before, from the query

The full rule based archetype distribution built in the baseline notebook gets printed here, before KMeans ever saw the data. This is the claim being checked, it was derived from explicit thresholds on the raw warehouse query, not from any model. INSUFFICIENT_DATA and monitor dominate the full page level population, with rising_stars, weak_no_demand_pages, champions, and engagement_problem_pages as the next largest named groups.

In [2]:
print("claim before, full rule based distribution, all pages:")
print(page_level["archetype"].value_counts())

claim before, full rule based distribution, all pages:
archetype
INSUFFICIENT_DATA           272549
monitor                      56390
rising_stars                 19211
weak_no_demand_pages         17426
champions                    14300
engagement_problem_pages     12752
hidden_gems                   2271
stale_visible_pages             27
cannibalization_risk             2
Name: count, dtype: int64


## Claim after, from the model

The archetype_kmeans distribution produced by the final k=20 model in the clustering notebook gets printed here. This is the claim made after running an unsupervised model on five behavioral features, with no access to the rule based thresholds. needs_review is the largest group, since the model declines to name an archetype for any cluster whose plurality vote is weak, followed by weak_no_demand_pages, rising_stars, engagement_problem_pages, champions, and hidden_gems.

In [3]:
print("claim after, the model's archetype distribution, train set:")
print(train["archetype_kmeans"].value_counts())

claim after, the model's archetype distribution, train set:
archetype_kmeans
needs_review                47510
weak_no_demand_pages        20462
rising_stars                18581
engagement_problem_pages    11536
champions                    8181
hidden_gems                  1253
Name: count, dtype: int64


## Verifying the after claim against the before claim

The rule based archetype gets cross tabulated here against the final archetype_kmeans for every train page. Full agreement is not the goal, disagreement is informative on its own, since it can mean the rule based thresholds drew lines the real feature space does not support, or that the model is finding a genuinely different pattern. overlap shows how each rule based category is actually spread across the model's final archetypes, the direct evidence behind every claim made in the clustering notebook.

In [4]:
overlap = pd.crosstab(train["archetype"], train["archetype_kmeans"])
overlap

archetype_kmeans,champions,engagement_problem_pages,hidden_gems,needs_review,rising_stars,weak_no_demand_pages
archetype,,,,,,
cannibalization_risk,0,1,0,1,0,0
champions,4032,1572,49,821,5581,0
engagement_problem_pages,125,4585,0,4189,1508,714
hidden_gems,190,0,672,0,1176,3
monitor,59,4482,0,32780,1465,11396
rising_stars,3775,896,134,2163,8815,49
stale_visible_pages,0,0,1,0,0,8
weak_no_demand_pages,0,0,397,7556,36,8292


## Validation metrics, baseline claim against model claim

The model's clustering and the rule based labels, treated as if they were a clustering, get scored here on the identical scaled feature space, with silhouette, Davies Bouldin, and Calinski Harabasz, alongside a random_floor column, the same 20 cluster ids assigned with no signal at all, so both real claims have an actual base rate to clear rather than just each other. The Adjusted Rand Index and Normalized Mutual Information get reported between the two labelings too, and those measure agreement, not quality, so they stay separate from the three quality metrics. Both the rule based baseline and the model clear the random floor by a wide margin on every metric, which is the real headline, not just which of the two beats the other by more, and the model wins the direct comparison on all three quality measures while agreement with the rule based baseline stays low, meaning the after claim is not simply restating the before claim.

In [5]:
scaler = RobustScaler()
X_train = scaler.fit_transform(train[FEATURE_COLS])
train_clusters = train["kmeans_cluster"].values

def cluster_quality(X, labels):
    return {
        "silhouette": silhouette_score(X, labels, sample_size=SIL_SAMPLE_SIZE, random_state=42),
        "davies_bouldin": davies_bouldin_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
    }

model_quality = cluster_quality(X_train, train_clusters)

baseline_mask = (train["archetype"] != "cannibalization_risk").values
baseline_labels = train["archetype"].values
baseline_quality = cluster_quality(X_train[baseline_mask], baseline_labels[baseline_mask])

# the floor: the same 20 cluster ids assigned with no signal at all, so the other two
# columns have an actual base rate to clear, not just each other
random_labels = np.random.RandomState(42).randint(0, train_clusters.max() + 1, size=len(train_clusters))
random_quality = cluster_quality(X_train, random_labels)

quality_compare = pd.DataFrame({
    "random_floor": random_quality,
    "baseline_rule_based": baseline_quality,
    "model_kmeans": model_quality,
})
quality_compare["higher_is_better"] = quality_compare.index != "davies_bouldin"
quality_compare["winner"] = np.where(
    quality_compare["higher_is_better"],
    np.where(quality_compare["model_kmeans"] > quality_compare["baseline_rule_based"], "model", "baseline"),
    np.where(quality_compare["model_kmeans"] < quality_compare["baseline_rule_based"], "model", "baseline"),
)
print(quality_compare)

print()
print("agreement between the two claims, not a quality score, just overlap:")
print(f"  Adjusted Rand Index:    {adjusted_rand_score(baseline_labels[baseline_mask], train_clusters[baseline_mask]):.3f}")
print(f"  Normalized Mutual Info: {normalized_mutual_info_score(baseline_labels[baseline_mask], train_clusters[baseline_mask]):.3f}")

                   random_floor  baseline_rule_based  model_kmeans  \
silhouette            -0.070096             0.008006      0.326640   
davies_bouldin       280.459075             3.050598      0.897309   
calinski_harabasz      0.892731          6046.314874  64753.199708   

                   higher_is_better winner  
silhouette                     True  model  
davies_bouldin                False  model  
calinski_harabasz              True  model  

agreement between the two claims, not a quality score, just overlap:


  Adjusted Rand Index:    0.098


  Normalized Mutual Info: 0.285


## What the model leans on

KMeans has no coefficients to read off the way a fitted supervised model would, so there is no feature_importances_ attribute to print. What stands in for it here is an ANOVA F-statistic per feature, the same test a one-way analysis of variance runs, comparing how much a feature's mean moves between the 20 clusters against how much it varies inside a single cluster. A feature the clustering leans on heavily should differ sharply from cluster to cluster and stay comparatively tight within any one cluster, a feature it barely uses should look about the same everywhere. This says which features are doing the separating, it does not say the clustering is right to separate on them.

In [6]:
f_stats, _ = f_classif(X_train, train_clusters)
feature_lean = pd.DataFrame({"feature": FEATURE_COLS, "f_statistic": f_stats}).sort_values("f_statistic", ascending=False)
feature_lean

,feature,f_statistic
2,days_since_last_update,124335.554023
0,ctr,113095.161515
1,avg_position,18871.907530
3,total_impressions_log,9245.395028
4,engagement_rate_filled,106.580276


## Where the model is wrong

A per page silhouette score gets computed here on a fixed random sample of train, computing it for every row would be too expensive, and the 20 worst fitting pages get listed. A negative silhouette means a page sits closer to a neighboring cluster than to its own, and reading the actual worst cases, not just an average number, is how a genuine misfit gets told apart from noise.

In [7]:
# silhouette_samples is also O(n^2); score a fixed random subsample of train rather than all rows
np.random.seed(42)
sil_idx = np.random.choice(len(train), size=min(SIL_SAMPLE_SIZE, len(train)), replace=False)
sil_sample = train.iloc[sil_idx].copy()
sil_sample["silhouette"] = silhouette_samples(X_train[sil_idx], train_clusters[sil_idx])

print(f"negative silhouette: {(sil_sample['silhouette'] < 0).sum()} rows, {(sil_sample['silhouette'] < 0).mean()*100:.1f}% (of {len(sil_sample)}-row sample)")
print(f"silhouette under 0.1: {(sil_sample['silhouette'] < 0.1).sum()} rows, {(sil_sample['silhouette'] < 0.1).mean()*100:.1f}%")

worst_fits = sil_sample.sort_values("silhouette").head(20)
worst_fits[["client_hash_id", "content_hash_id", "archetype_kmeans", "kmeans_cluster", "silhouette"] + RAW_COLS]

negative silhouette: 125 rows, 2.5% (of 5000-row sample)
silhouette under 0.1: 588 rows, 11.8%


,client_hash_id,content_hash_id,archetype_kmeans,kmeans_cluster,silhouette,total_impressions,total_clicks,ctr,avg_position,days_since_last_update
77744,client_2094c6eb080311d5,content_6772be363d1c4720,hidden_gems,2,-0.181909,38.0,4.0,0.105263,13.644737,41
92390,client_8ddc46da5414ffd8,content_18fadade2c33e046,rising_stars,6,-0.160403,264.0,3.0,0.011364,21.014142,29
62859,client_0fa64a184f18a4a0,content_add3320f300eb338,rising_stars,6,-0.159167,86.0,1.0,0.011628,3.544813,41
44367,client_73cda7b4e4f265ea,content_5d4fc14ee9921162,rising_stars,6,-0.145745,496.0,6.0,0.012097,5.723504,41
62917,client_0fa64a184f18a4a0,content_991c5b108377e826,rising_stars,6,-0.141565,168.0,2.0,0.011905,2.002684,41
83822,client_e00b29e582949543,content_df958dbaea04d57d,rising_stars,6,-0.135600,173.0,2.0,0.011561,4.961559,29
86712,client_3f0ce4d44fe94f3d,content_a43a2fa1fddb6430,rising_stars,6,-0.134335,85.0,1.0,0.011765,9.126540,41
40258,client_73cda7b4e4f265ea,content_91e0c3cdb310eb7c,rising_stars,6,-0.126141,654.0,8.0,0.012232,15.643380,43
27963,client_e5c2aa26a8598242,content_c63439e949028d34,rising_stars,6,-0.112502,230.0,3.0,0.013043,18.349375,8
90772,client_8ddc46da5414ffd8,content_440c8a356cc0f4eb,rising_stars,6,-0.112093,87.0,1.0,0.011494,10.472250,29


Fifteen of the twenty worst fitting pages sit in the same cluster, cluster 6, the one plurality voted rising_stars out of its roughly 4200 members. Their CTR is nearly identical, clustered tight around 0.011 to 0.013, but their avg_position spans from about 2 to about 27, top of page one to bottom of page three. CTR is doing most of the work of holding that cluster together and position is not, so a page at either end of that position range sits about as close to a neighboring cluster as to its own cluster 6 siblings, which is exactly what a negative silhouette means and exactly what a cluster built mostly on one dominant feature looks like when read at the individual page level instead of the average level.

One page, client_2094c6eb080311d5's content_6772be363d1c4720, lands in a small hidden_gems cluster on a 10.5 percent CTR from only 38 impressions, but its avg_position of 13.6 sits just past the good_position cutoff of 10 the rule engine itself uses. The rule engine would not have called this page a hidden gem at all, its own threshold says no, while the clustering's continuous distance calculation pulled it in anyway on the strength of that CTR. Low volume makes a single extra click swing the CTR a lot, so this reads as a real boundary case, not an obvious model error, worth a person's eyes before it gets protected or improved on the strength of a clustering call alone.

client_73cda7b4e4f265ea's content_0ecfd07cb87a5c4c is uncertain twice over: it sits in a needs_review cluster, meaning the group level naming confidence check already declined to name it, and it also carries a negative silhouette, meaning the individual fit check flags it too. Zero clicks against 94 impressions and a stale 125 days since update do not obviously belong to any of the six named archetypes, which is arguably the correct outcome here, needs_review doing its job on a page that the rule engine's own thresholds do not cleanly describe either.

## Conclusion

The before claim and the after claim are not the same statement wearing two labels. The model beats the rule based baseline as a clustering on every internal quality measure, and both clear the random floor by a wide margin, yet agreement between the baseline and the model stays low. The honest reading is that the model found real structure in the same five signals the rules use, but drew different boundaries than a person writing thresholds by hand would have drawn, leaning most on days_since_last_update and CTR, with avg_position a clear third and total_impressions_log and engagement_rate_filled contributing the least. That ranking is also part of why its most uncertain pages concentrate in a cluster where CTR stays flat but position keeps moving, position separating clusters far less cleanly than CTR does. Neither claim has been checked against a real world outcome such as ranking movement, so both stay observed and directional, not certified.